# Multi-Tier Memory System

**Multigen SDK — Notebook 21**

---

## Overview

Human cognition is organized into distinct memory systems: a fast, volatile working
memory for active tasks; a rich episodic store of past experiences; and a vast
semantic store of facts and relationships.  The Multigen memory system mirrors this
architecture with four cooperating tiers:

| Tier | Class | Analogy | Typical use |
|------|-------|---------|-------------|
| Short-term | `ShortTermMemory` | Scratchpad | Per-request values, intermediate results |
| Episodic | `EpisodicMemory` | Diary | Ordered log of agent interactions |
| Working | `WorkingMemory` | Whiteboard | Rolling context window fed to LLMs |
| Semantic | `SemanticMemory` | Encyclopedia | Long-lived facts with optional vector search |

`MemoryManager` is a unified facade that wires all four tiers together and provides
high-level operations: `remember`, `recall`, `inject_context`, and `consolidate`.

## What this notebook covers

| Section | Topic |
|---------|-------|
| 1 | Setup and imports |
| 2 | ShortTermMemory — store, retrieve, LRU eviction, tag search |
| 3 | TTL expiry demonstration |
| 4 | EpisodicMemory — record, recent, by_agent, keyword search |
| 5 | WorkingMemory — push/pop/peek, summarize for LLM injection |
| 6 | SemanticMemory — keyword fallback search, top_k |
| 7 | SemanticMemory with custom embedding function |
| 8 | MemoryManager facade — remember/recall/inject_context |
| 9 | Memory consolidation — short-term → semantic promotion |
| 10 | Agent workflow with memory accumulation |
| 11 | Memory isolation between sessions |
| 12 | Summary table |

All code is pure in-memory.  No external APIs or databases are required.


## Section 1 — Setup and Imports

The memory classes live in `multigen.memory` and are re-exported from the top-level
`multigen` package.  We import all five public names:

- `ShortTermMemory` — volatile session store with optional TTL and LRU eviction
- `EpisodicMemory` — append-only log of agent interaction episodes
- `WorkingMemory` — bounded sliding window for context injection
- `SemanticMemory` — long-lived factual store with pluggable embedding search
- `MemoryManager` — unified facade over all four tiers


In [ ]:
import asyncio
import os
import sys
import time

# ── SDK path resolution ────────────────────────────────────────────────────────
for candidate in ['../sdk', 'sdk']:
    candidate = os.path.abspath(candidate)
    if os.path.isdir(candidate) and candidate not in sys.path:
        sys.path.insert(0, candidate)

from multigen import (
    ShortTermMemory,
    EpisodicMemory,
    WorkingMemory,
    SemanticMemory,
    MemoryManager,
)

print('ShortTermMemory :', ShortTermMemory)
print('EpisodicMemory  :', EpisodicMemory)
print('WorkingMemory   :', WorkingMemory)
print('SemanticMemory  :', SemanticMemory)
print('MemoryManager   :', MemoryManager)
print()
print('All imports successful.')


## Section 2 — ShortTermMemory

`ShortTermMemory` is a session-scoped in-memory key/value store.  Key features:

- **Capacity + LRU eviction** — when the store is full, the least-recently-used
  entry is evicted to make room for the new one.
- **Per-entry TTL** — each entry can have an optional expiry time (seconds).
  Expired entries are lazily removed on access and eagerly swept before insertions.
- **Tag-based search** — entries can be tagged at write time; `search_by_tags` returns
  all matching (key, value) pairs without touching access timestamps.
- **Stats** — `stats()` returns current size, capacity, and utilization ratio.

### Constructor parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `capacity` | 1000 | Maximum number of live entries |
| `default_ttl` | `None` | Default TTL in seconds; `None` = immortal |


In [ ]:
# ── Create a small store to make LRU eviction easy to observe ─────────────────

stm = ShortTermMemory(capacity=5, default_ttl=None)

# Store five entries with tags
stm.store('query',        'What is the capital of France?',   tags=['input', 'nlp'])
stm.store('language',     'en',                               tags=['meta'])
stm.store('session_id',   'sess-abc123',                      tags=['meta'])
stm.store('raw_response', 'Paris is the capital.',            tags=['output'])
stm.store('confidence',   0.97,                               tags=['output', 'metric'])

print('After storing 5 entries:')
print('  keys       :', stm.keys())
print('  len(stm)   :', len(stm))
print('  stats      :', stm.stats())
print()

# Retrieve a value — updates access timestamp for LRU ordering
val = stm.retrieve('query')
print(f'retrieve("query")   → {val!r}')

# Search by tag
meta_items  = stm.search_by_tags(['meta'])
input_items = stm.search_by_tags(['input'])
print(f'search_by_tags(["meta"])  → {meta_items}')
print(f'search_by_tags(["input"]) → {input_items}')
print()

# Force LRU eviction: store a 6th entry (capacity=5)
# The least-recently-used entry will be evicted.
# "language" was stored early and never retrieved → likely LRU candidate.
stm.store('new_key', 'this entry triggers LRU eviction', tags=['new'])

print('After storing 6th entry (capacity=5 → eviction):')
print('  keys   :', stm.keys())
print('  stats  :', stm.stats())
print()

# Demonstrate delete
deleted = stm.delete('confidence')
print(f'delete("confidence") returned: {deleted}')
print(f'retrieve("confidence") after delete: {stm.retrieve("confidence")!r}')


## Section 3 — TTL Expiry Demonstration

Each entry in `ShortTermMemory` can carry an individual TTL (time-to-live) in
seconds.  When you call `retrieve` after the TTL has elapsed the entry is removed
and `None` is returned.

The `default_ttl` constructor argument sets a fallback for entries that don't
specify their own TTL.  Passing `ttl=` to `store()` overrides the default for that
specific entry.

This section stores three entries:

- one **immortal** (no TTL),
- one **short-lived** (`ttl=0.1` — expires after 100 ms),
- one that uses the **store-level default TTL**.

We then sleep for 200 ms and verify that the short-lived entry has expired while
the immortal one is still alive.


In [ ]:
stm_ttl = ShortTermMemory(capacity=100, default_ttl=60.0)  # 60-second default

# Entry 1: immortal (no TTL — overrides the 60-second default)
stm_ttl.store('permanent_fact', 'The Earth orbits the Sun', ttl=None)

# Entry 2: short-lived — expires after 100 ms
stm_ttl.store('temp_token', 'tok-xyz', ttl=0.1)

# Entry 3: uses store-level default (60 s)
stm_ttl.store('session_ctx', {'user': 'alice', 'locale': 'en-US'})

print('Immediately after storing:')
print(f'  permanent_fact : {stm_ttl.retrieve("permanent_fact")!r}')
print(f'  temp_token     : {stm_ttl.retrieve("temp_token")!r}')
print(f'  session_ctx    : {stm_ttl.retrieve("session_ctx")!r}')
print()

# Wait for the short-lived entry to expire
time.sleep(0.2)

print('After sleeping 0.2 s (temp_token TTL was 0.1 s):')
print(f'  permanent_fact : {stm_ttl.retrieve("permanent_fact")!r}  ← still alive')
print(f'  temp_token     : {stm_ttl.retrieve("temp_token")!r}      ← expired → None')
print(f'  session_ctx    : {stm_ttl.retrieve("session_ctx")!r}  ← alive (60 s default TTL)')
print()
print(f'Store size (expired entries auto-removed): {len(stm_ttl)}')


## Section 4 — EpisodicMemory

`EpisodicMemory` maintains an **ordered log** of agent interaction episodes.  It is
backed by a `collections.deque` with a configurable `max_episodes` bound — the
oldest episodes are automatically dropped when the buffer is full.

Each `Episode` records:

| Field | Description |
|-------|-------------|
| `agent` | Name of the agent that produced the episode |
| `inputs` | Dict of inputs passed to the agent |
| `outputs` | Dict of outputs returned by the agent |
| `episode_id` | UUID (auto-generated) |
| `timestamp` | Unix timestamp (auto-generated) |
| `tags` | Optional list of string labels |
| `metadata` | Optional free-form dict |

### Key methods

| Method | Description |
|--------|-------------|
| `record(agent, inputs, outputs)` | Append a new episode, return its `episode_id` |
| `recent(n)` | Return the last N episodes (newest last) |
| `by_agent(name)` | Return all episodes for a specific agent |
| `search(query)` | Keyword search across inputs, outputs, and tags |


In [ ]:
epis = EpisodicMemory(max_episodes=200)

# Simulate a 3-step agent pipeline producing episodes
ep1 = epis.record(
    agent='DataFetchAgent',
    inputs={'source': 'sales_db', 'limit': 100},
    outputs={'rows': 87, 'schema': ['id', 'amount', 'date']},
    tags=['fetch', 'database'],
)

ep2 = epis.record(
    agent='DataCleanAgent',
    inputs={'rows': 87},
    outputs={'clean_rows': 82, 'dropped': 5, 'reason': 'null_values'},
    tags=['clean', 'preprocessing'],
)

ep3 = epis.record(
    agent='AnalysisAgent',
    inputs={'clean_rows': 82},
    outputs={'mean_amount': 1234.56, 'total': 101333.92, 'insight': 'Q4 spike detected'},
    tags=['analysis', 'statistics'],
)

ep4 = epis.record(
    agent='DataFetchAgent',
    inputs={'source': 'inventory_db', 'limit': 50},
    outputs={'rows': 50, 'schema': ['product_id', 'qty', 'location']},
    tags=['fetch', 'database'],
)

print(f'Total episodes recorded: {len(epis)}')
print()

# recent(n)
print('Last 2 episodes (most recent last):')
for ep in epis.recent(2):
    print(f'  [{ep.agent}] inputs={ep.inputs} → outputs={ep.outputs}')
print()

# by_agent
fetch_eps = epis.by_agent('DataFetchAgent')
print(f'Episodes from DataFetchAgent: {len(fetch_eps)}')
for ep in fetch_eps:
    print(f'  episode_id={ep.episode_id[:8]}...  source={ep.inputs.get("source")}')
print()

# keyword search
search_results = epis.search('spike')
print(f'search("spike") → {len(search_results)} result(s):')
for ep in search_results:
    print(f'  [{ep.agent}] outputs={ep.outputs}')


## Section 5 — WorkingMemory

`WorkingMemory` is a **bounded sliding window** — a deque with a fixed `maxsize`.
When the window is full, pushing a new item automatically drops the oldest one.
This matches the cognitive "working memory" metaphor: you can actively hold only a
limited number of items at once.

### Typical use: rolling LLM conversation history

Each turn of a conversation (user message + assistant reply) is pushed as a dict.
Before calling the LLM, `summarize()` concatenates all items into a single string
that can be pasted directly into a prompt.

### Methods

| Method | Description |
|--------|-------------|
| `push(item)` | Append an item; evict the oldest if full |
| `pop()` | Remove and return the most recent item |
| `peek(n)` | Return the last N items without modifying the buffer |
| `all()` | Return a copy of the entire buffer |
| `summarize(separator)` | Join all items as strings — for LLM context injection |
| `clear()` | Empty the buffer |


In [ ]:
wm = WorkingMemory(maxsize=4)

# Simulate a conversation turn-by-turn
messages = [
    {'role': 'user',      'content': 'What is 2 + 2?'},
    {'role': 'assistant', 'content': 'That equals 4.'},
    {'role': 'user',      'content': 'And 4 * 3?'},
    {'role': 'assistant', 'content': 'That equals 12.'},
]

for msg in messages:
    wm.push(msg)

print(f'Working memory size (maxsize=4): {len(wm)}')
print('Current buffer:')
for item in wm.all():
    print(f'  {item}')
print()

# Push a 5th item — the oldest (messages[0]) is dropped
wm.push({'role': 'user', 'content': 'Now what is 12 / 4?'})
print(f'After 5th push (oldest evicted):')
print(f'  size: {len(wm)}')
for item in wm.all():
    print(f'  {item}')
print()

# peek() — last 2 items only
print(f'peek(2): {wm.peek(2)}')
print()

# summarize() — join as a string for LLM prompt injection
ctx_string = wm.summarize()
print('summarize() output (for LLM context injection):')
print('---')
print(ctx_string)
print('---')
print()

# pop() — removes most recent
last = wm.pop()
print(f'pop() → {last}')
print(f'After pop, size: {len(wm)}')


## Section 6 — SemanticMemory (Keyword Fallback)

`SemanticMemory` is a **long-lived factual store** with two retrieval modes:

1. **Keyword fallback** (default, no embedding function) — each fact is scored by
   how many query words appear in its `description` + tags text.
2. **Embedding search** (Section 7) — if you supply an `embedding_fn`, `search`
   computes cosine similarity against stored fact vectors.

### Key methods

| Method | Description |
|--------|-------------|
| `store_fact(key, fact, description, tags)` | Store a named fact |
| `retrieve_fact(key)` | Direct key lookup |
| `search(query, top_k)` | Return `[(key, fact, score)]` sorted by relevance |
| `forget(key)` | Remove a fact |
| `all_keys()` | List all stored keys |

The `description` is the text indexed for search; the `fact` is the opaque payload
returned to the caller.


In [ ]:
sem = SemanticMemory()  # no embedding_fn → keyword fallback

# Store a variety of facts
sem.store_fact('earth_radius',   6371,                   description='radius of the Earth in kilometres', tags=['geography', 'planet'])
sem.store_fact('speed_of_light', 299_792_458,            description='speed of light in metres per second', tags=['physics', 'constants'])
sem.store_fact('paris_capital',  'Paris',                description='Paris is the capital city of France', tags=['geography', 'capital', 'europe'])
sem.store_fact('python_creator', 'Guido van Rossum',     description='Guido van Rossum created Python programming language', tags=['programming', 'python'])
sem.store_fact('boiling_water',  100,                    description='boiling point of water in Celsius at sea level', tags=['chemistry', 'physics'])
sem.store_fact('haversine',      'distance formula',     description='haversine formula computes great-circle distance between two points on Earth', tags=['geography', 'maths'])

print(f'Facts stored: {len(sem)}')
print(f'All keys: {sem.all_keys()}')
print()

# Direct lookup
print(f'retrieve_fact("earth_radius") → {sem.retrieve_fact("earth_radius")}')
print()

# Keyword search
queries = ['capital France', 'physics constants', 'Earth geography', 'Python programming']
for q in queries:
    results = sem.search(q, top_k=3)
    print(f'search({q!r}):')
    for key, fact, score in results:
        print(f'  key={key!r:<20} score={score:.1f}  fact={fact!r}')
    print()


## Section 7 — SemanticMemory with Custom Embedding Function

When an `embedding_fn` is provided, `SemanticMemory` switches from keyword scoring
to **cosine similarity** over dense vectors.  This enables proper semantic search:
"French capital city" can match "Paris is the capital of France" even though the
words don't overlap exactly.

The embedding function must have signature `(text: str) -> List[float]`.

For this demonstration we use a simple **bag-of-words** mock that produces a sparse
vector over a fixed vocabulary — enough to illustrate the API without requiring any
external model.


In [ ]:
from typing import List

# ── Bag-of-words mock embedding ────────────────────────────────────────────────
# Vocabulary: top-30 words likely to appear in our facts.
_VOCAB = [
    'capital', 'city', 'france', 'paris', 'europe', 'radius', 'earth', 'kilometre',
    'geography', 'speed', 'light', 'metre', 'second', 'physics', 'constant',
    'python', 'programming', 'language', 'guido', 'creator', 'boiling', 'water',
    'celsius', 'chemistry', 'haversine', 'distance', 'formula', 'great', 'circle', 'point',
]

def bow_embed(text: str) -> List[float]:
    """Return a normalised bag-of-words vector over the fixed vocabulary."""
    words = set(text.lower().split())
    vec = [1.0 if w in words else 0.0 for w in _VOCAB]
    norm = sum(v ** 2 for v in vec) ** 0.5
    return [v / norm for v in vec] if norm > 0 else vec

# Verify the embedding function
sample_vec = bow_embed('Paris is the capital city of France')
print(f'Embedding dim: {len(sample_vec)}')
non_zero = [(w, v) for w, v in zip(_VOCAB, sample_vec) if v > 0]
print(f'Non-zero dimensions: {non_zero}')
print()

# ── SemanticMemory with the custom embedding function ──────────────────────────
sem_emb = SemanticMemory(embedding_fn=bow_embed)

sem_emb.store_fact('earth_radius',   6371,                 description='radius of the Earth in kilometres', tags=['geography'])
sem_emb.store_fact('speed_of_light', 299_792_458,          description='speed of light metre second physics constant', tags=['physics'])
sem_emb.store_fact('paris_capital',  'Paris',              description='Paris capital city France Europe', tags=['geography'])
sem_emb.store_fact('python_creator', 'Guido van Rossum',   description='Guido creator Python programming language', tags=['programming'])
sem_emb.store_fact('boiling_water',  100,                  description='boiling water Celsius chemistry', tags=['chemistry'])
sem_emb.store_fact('haversine',      'distance formula',   description='haversine distance formula great circle Earth point', tags=['geography', 'maths'])

# Search with queries whose words don't overlap directly with descriptions
embedding_queries = [
    'French capital city',          # should match paris_capital
    'planet radius kilometre',      # should match earth_radius + haversine
    'physics speed constant',       # should match speed_of_light
]

for q in embedding_queries:
    results = sem_emb.search(q, top_k=3)
    print(f'Embedding search({q!r}):')
    for key, fact, score in results:
        print(f'  key={key!r:<20} cosine={score:.4f}  fact={fact!r}')
    print()


## Section 8 — MemoryManager Facade

`MemoryManager` is the **single entry-point** for multi-tier memory.  It owns one
instance of each tier and exposes a simplified API:

| Method | Description |
|--------|-------------|
| `remember(key, value, tier, **kwargs)` | Store in `"short"`, `"semantic"`, or `"working"` |
| `recall(key, tier)` | Retrieve from `"short"` or `"semantic"` |
| `record_episode(agent, inputs, outputs)` | Append to episodic memory |
| `search(query, top_k)` | Semantic search across factual memory |
| `inject_context(ctx)` | Enrich a context dict with working memory and recent episodes |
| `consolidate()` | Promote all short-term entries to semantic memory |
| `stats()` | Return size statistics for all four tiers |

You can also access the individual tiers directly via
`mgr.short_term`, `mgr.episodic`, `mgr.working`, and `mgr.semantic`.


In [ ]:
mgr = MemoryManager()

# ── remember / recall across tiers ────────────────────────────────────────────

# Short-term tier
mgr.remember('user_query', 'What is the GDP of Germany?', tier='short', tags=['query'])
mgr.remember('session_locale', 'de-DE', tier='short', tags=['meta'])

# Semantic tier
mgr.remember('germany_gdp', 4.07e12, tier='semantic',
             description='GDP of Germany in USD for 2023', tags=['economics', 'germany'])
mgr.remember('france_gdp', 2.78e12, tier='semantic',
             description='GDP of France in USD for 2023', tags=['economics', 'france'])

# Working tier
mgr.working.push({'role': 'user',      'content': 'What is the GDP of Germany?'})
mgr.working.push({'role': 'assistant', 'content': 'Germany GDP is approximately 4.07 trillion USD.'})

print('Short-term recall:')
print(f'  recall("user_query",     tier="short") → {mgr.recall("user_query",     tier="short")!r}')
print(f'  recall("session_locale", tier="short") → {mgr.recall("session_locale", tier="short")!r}')
print()

print('Semantic recall:')
print(f'  recall("germany_gdp", tier="semantic") → {mgr.recall("germany_gdp", tier="semantic")}')
print()

print('Semantic search("Germany economics"):')
for key, fact, score in mgr.search('Germany economics', top_k=3):
    print(f'  key={key!r:<14}  score={score:.2f}  fact={fact}')
print()

# ── inject_context ─────────────────────────────────────────────────────────────
# Record an episode first so inject_context has something to show

mgr.record_episode(
    'GDPAgent',
    inputs={'query': 'GDP Germany'},
    outputs={'gdp_usd': 4.07e12, 'year': 2023},
)

base_ctx = {'task': 'economic analysis', 'region': 'europe'}
enriched = mgr.inject_context(base_ctx)

print('inject_context enrichment keys:', list(enriched.keys()))
print('  _working_memory    :', enriched['_working_memory'])
print('  _recent_episodes   :', enriched['_recent_episodes'])
print()
print('Stats:')
for tier, info in mgr.stats().items():
    print(f'  {tier:<12}: {info}')


## Section 9 — Memory Consolidation

`consolidate()` simulates a **memory consolidation pass** — the process by which
short-lived working knowledge is promoted into long-term semantic memory.  It:

1. Iterates all unexpired keys in `ShortTermMemory`.
2. Calls `SemanticMemory.store_fact(key, value, description=str(value))` for each.
3. Clears `ShortTermMemory`.
4. Returns the number of entries promoted.

This is useful when you want short-term scratchpad values to "survive" session
boundaries without requiring the calling code to decide what to keep.


In [ ]:
mgr2 = MemoryManager()

# Load short-term memory with several scratchpad values
mgr2.remember('pipeline_run_id', 'run-2024-0317-001', tier='short', tags=['run'])
mgr2.remember('cleaned_rows',    2847,                 tier='short', tags=['stats'])
mgr2.remember('dropped_rows',    153,                  tier='short', tags=['stats'])
mgr2.remember('mean_spend',      429.30,               tier='short', tags=['metrics'])
mgr2.remember('p99_latency_ms',  312,                  tier='short', tags=['metrics'])

print('Before consolidation:')
before_stats = mgr2.stats()
print(f'  short_term : {before_stats["short_term"]}')
print(f'  semantic   : {before_stats["semantic"]}')
print()

# Run consolidation
promoted = mgr2.consolidate()
print(f'consolidate() promoted {promoted} entries.')
print()

print('After consolidation:')
after_stats = mgr2.stats()
print(f'  short_term : {after_stats["short_term"]}')
print(f'  semantic   : {after_stats["semantic"]}')
print()

# Verify the promoted values are retrievable via semantic recall
print('Semantic recall after consolidation:')
for key in ['pipeline_run_id', 'cleaned_rows', 'mean_spend']:
    val = mgr2.recall(key, tier='semantic')
    print(f'  recall({key!r:<22}, tier="semantic") → {val!r}')
print()

# Short-term entries are now gone
print('Short-term recall after consolidation:')
for key in ['pipeline_run_id', 'cleaned_rows']:
    val = mgr2.recall(key, tier='short')
    print(f'  recall({key!r:<22}, tier="short")    → {val!r}  ← cleared')


## Section 10 — Agent Workflow with Memory Accumulation

This section demonstrates a realistic multi-step agent chain where each agent
reads from and writes to a shared `MemoryManager`.  The workflow:

```
StepA (research)  →  StepB (analysis)  →  StepC (report)
```

Each step:
1. Looks up prior context via `recall` and `search`.
2. Performs its "work" (mocked as a pure function).
3. Records its episode and writes results back to memory.

At the end, `inject_context` collects the accumulated working memory and recent
episodes — ready to be fed as context to any subsequent LLM call.


In [ ]:
from multigen import FunctionAgent

pipeline_mem = MemoryManager()

# ── Step A: ResearchAgent ──────────────────────────────────────────────────────

async def research_step(ctx: dict) -> dict:
    topic = ctx.get('topic', 'unknown')
    # Simulate research output
    findings = {
        'topic': topic,
        'sources': ['arxiv:2401.01234', 'arxiv:2401.05678'],
        'summary': f'Recent advances in {topic} show 43% improvement in benchmarks.',
        'key_entities': [topic, 'neural networks', 'transformers'],
    }
    # Store findings in memory
    pipeline_mem.remember(f'research_{topic}', findings, tier='short', tags=['research'])
    pipeline_mem.working.push({'step': 'research', 'summary': findings['summary']})
    pipeline_mem.record_episode('ResearchAgent',
                                inputs={'topic': topic},
                                outputs=findings,
                                tags=['research'])
    return findings

research_agent = FunctionAgent('ResearchAgent', fn=research_step)

# ── Step B: AnalysisAgent ──────────────────────────────────────────────────────

async def analysis_step(ctx: dict) -> dict:
    topic = ctx.get('topic', 'unknown')
    # Pull research findings from memory
    findings = pipeline_mem.recall(f'research_{topic}', tier='short') or {}
    n_sources = len(findings.get('sources', []))
    analysis = {
        'topic': topic,
        'source_count': n_sources,
        'confidence': 0.85 if n_sources >= 2 else 0.50,
        'recommendation': f'Proceed with {topic} implementation — confidence 85%',
    }
    pipeline_mem.remember(f'analysis_{topic}', analysis, tier='short', tags=['analysis'])
    pipeline_mem.working.push({'step': 'analysis', 'confidence': analysis['confidence']})
    pipeline_mem.record_episode('AnalysisAgent',
                                inputs={'topic': topic, 'source_count': n_sources},
                                outputs=analysis,
                                tags=['analysis'])
    return analysis

analysis_agent = FunctionAgent('AnalysisAgent', fn=analysis_step)

# ── Step C: ReportAgent ────────────────────────────────────────────────────────

async def report_step(ctx: dict) -> dict:
    topic = ctx.get('topic', 'unknown')
    # Enrich context with memory before composing report
    enriched = pipeline_mem.inject_context({'topic': topic})
    working_history = enriched['_working_memory']
    recent_episodes = enriched['_recent_episodes']

    report = {
        'title': f'Report: {topic}',
        'steps_completed': len(working_history),
        'episodes_logged': len(recent_episodes),
        'working_memory_snapshot': working_history,
        'final_recommendation': pipeline_mem.recall(f'analysis_{topic}', tier='short')
                                 .get('recommendation', 'N/A') if pipeline_mem.recall(f'analysis_{topic}', tier='short') else 'N/A',
    }
    pipeline_mem.record_episode('ReportAgent',
                                inputs={'topic': topic},
                                outputs=report,
                                tags=['report'])
    return report

report_agent = FunctionAgent('ReportAgent', fn=report_step)

# ── Run the chain ──────────────────────────────────────────────────────────────

ctx = {'topic': 'multi-agent LLM systems'}

research_out  = await research_agent.run(ctx)
analysis_out  = await analysis_agent.run(ctx)
report_out    = await report_agent.run(ctx)

print('Research output:')
print(f'  summary  : {research_out["summary"]}')
print(f'  sources  : {research_out["sources"]}')
print()
print('Analysis output:')
print(f'  confidence     : {analysis_out["confidence"]}')
print(f'  recommendation : {analysis_out["recommendation"]}')
print()
print('Report output:')
for k, v in report_out.items():
    print(f'  {k:<28} : {v}')
print()
print('Final memory stats:')
for tier, info in pipeline_mem.stats().items():
    print(f'  {tier:<12}: {info}')


## Section 11 — Memory Isolation Between Sessions

Each `MemoryManager` instance owns its own tier objects.  Two managers created
independently share no state — writes to one are never visible from the other.

This mirrors real deployment scenarios: each user session, each agent run, or each
tenant gets its own `MemoryManager`, ensuring complete isolation without any
locking or namespace management overhead.


In [ ]:
# Create two completely independent MemoryManager instances
mgr_alice = MemoryManager()
mgr_bob   = MemoryManager()

# Alice stores her session data
mgr_alice.remember('preference',   'dark mode',      tier='short', tags=['ui'])
mgr_alice.remember('last_query',   'AI regulations', tier='short')
mgr_alice.working.push({'role': 'user', 'content': 'Tell me about AI regulations.'})
mgr_alice.record_episode('QueryAgent',
                          inputs={'user': 'alice', 'query': 'AI regulations'},
                          outputs={'response': 'Under review by EU AI Act.'})

# Bob stores his session data
mgr_bob.remember('preference', 'light mode', tier='short', tags=['ui'])
mgr_bob.remember('last_query', 'Rust async',  tier='short')
mgr_bob.working.push({'role': 'user', 'content': 'Explain Rust async/await.'})
mgr_bob.record_episode('QueryAgent',
                        inputs={'user': 'bob', 'query': 'Rust async'},
                        outputs={'response': 'Rust async uses futures and an executor.'})

print('=== Alice\'s memory ===')
print(f'  recall("preference") → {mgr_alice.recall("preference", tier="short")!r}')
print(f'  recall("last_query") → {mgr_alice.recall("last_query", tier="short")!r}')
print(f'  working memory       → {mgr_alice.working.all()}')
print(f'  episodic count       → {len(mgr_alice.episodic)}')
print(f'  episodes             → {[e.inputs for e in mgr_alice.episodic.recent()]}')
print()

print('=== Bob\'s memory ===')
print(f'  recall("preference") → {mgr_bob.recall("preference", tier="short")!r}')
print(f'  recall("last_query") → {mgr_bob.recall("last_query", tier="short")!r}')
print(f'  working memory       → {mgr_bob.working.all()}')
print(f'  episodic count       → {len(mgr_bob.episodic)}')
print(f'  episodes             → {[e.inputs for e in mgr_bob.episodic.recent()]}')
print()

# Cross-contamination checks
alice_sees_bobs_pref  = mgr_alice.recall('preference', tier='short') == 'light mode'
bob_sees_alice_pref   = mgr_bob.recall('preference',   tier='short') == 'dark mode'
alice_has_bobs_epis   = any('bob' in str(e.inputs) for e in mgr_alice.episodic.recent())
bob_has_alice_epis    = any('alice' in str(e.inputs) for e in mgr_bob.episodic.recent())

print('Cross-contamination checks:')
print(f'  Alice sees Bob\'s preference  → {alice_sees_bobs_pref}  (expected False)')
print(f'  Bob sees Alice\'s preference  → {bob_sees_alice_pref}  (expected False)')
print(f'  Alice has Bob\'s episodes     → {alice_has_bobs_epis}  (expected False)')
print(f'  Bob has Alice\'s episodes     → {bob_has_alice_epis}  (expected False)')
print()
assert not alice_sees_bobs_pref, 'Alice should not see Bob\'s preference'
assert not bob_sees_alice_pref,  'Bob should not see Alice\'s preference'
assert not alice_has_bobs_epis,  'Alice should not have Bob\'s episodes'
assert not bob_has_alice_epis,   'Bob should not have Alice\'s episodes'
print('All isolation assertions passed.')


## Section 12 — Summary

### Memory tier quick-reference

| Tier | Class | Capacity | TTL | Eviction | Primary use |
|------|-------|----------|-----|----------|-------------|
| Short-term | `ShortTermMemory` | Configurable (default 1000) | Optional per-entry | LRU | Per-request scratchpad, intermediate results |
| Episodic | `EpisodicMemory` | Configurable (default 500 episodes) | None (circular buffer) | FIFO on overflow | Audit log, conversation history |
| Working | `WorkingMemory` | Configurable (default 20 items) | None (bounded deque) | FIFO on overflow | Rolling LLM context window |
| Semantic | `SemanticMemory` | Unlimited | None | Never auto-evicted | Long-term facts, RAG knowledge base |

### MemoryManager API at a glance

| Method | Parameters | What it does |
|--------|-----------|--------------|
| `remember` | `(key, value, tier, **kwargs)` | Store in short / semantic / working tier |
| `recall` | `(key, tier)` | Retrieve from short or semantic tier |
| `search` | `(query, top_k)` | Semantic search over factual memory |
| `record_episode` | `(agent, inputs, outputs)` | Append to episodic log |
| `inject_context` | `(ctx)` | Enrich ctx dict with working memory + recent episodes |
| `consolidate` | `()` | Promote all short-term entries to semantic; clear short-term |
| `stats` | `()` | Return size info for all four tiers |

### Design patterns

```
# Pattern 1: per-request short-term scratchpad
mgr = MemoryManager()
mgr.remember("raw_input", data, tier="short")
# ... processing ...
result = mgr.recall("raw_input", tier="short")

# Pattern 2: LLM context window
mgr.working.push({"role": "user", "content": "..."})
mgr.working.push({"role": "assistant", "content": "..."})
prompt_ctx = mgr.working.summarize()

# Pattern 3: end-of-session consolidation
mgr.consolidate()  # short-term → semantic; cleared short-term

# Pattern 4: episodic audit
mgr.record_episode("MyAgent", inputs={...}, outputs={...})
last5 = mgr.episodic.recent(5)
```

### Next steps

- **Notebook 22**: Multi-Tier Caching System — LRU, TTL, async stampede protection
- Combine `MemoryManager` with `WorkflowDebugger` (Notebook 20) to capture
  memory state at each workflow snapshot.
- Replace the bag-of-words mock embedding with a real sentence-transformer to
  enable true semantic retrieval in `SemanticMemory`.
